# 03 — Data preprocessing

This notebook creates transparent, reversible analysis copies. Since there is no codebook, sentinel definitions, valid clinical ranges, or units, it does not recode `-1`, remove extreme observations, impute predictors, or alter target values. This stage audits parseability and documents cleaning decisions. Any model-time imputation must be fit only inside the training pipeline.

In [1]:
from pathlib import Path
import shutil,subprocess,tempfile
import numpy as np,pandas as pd
ROOT=Path.cwd()
if not (ROOT/"DATA.xls").exists(): ROOT=ROOT.parent
try: xls=pd.read_excel(ROOT/"DATA.xls",sheet_name=None)
except ImportError:
 exe=shutil.which("soffice") or "/Users/omvdangi/.cache/codex-runtimes/codex-primary-runtime/dependencies/bin/override/soffice"
 tmp=Path(tempfile.mkdtemp(prefix="myopia_nb_")); subprocess.run([exe,"--headless","--convert-to","xlsx","--outdir",str(tmp),str(ROOT/"DATA.xls")],check=True,capture_output=True); xls=pd.read_excel(tmp/"DATA.xlsx",sheet_name=None)
d3=pd.read_excel(ROOT/"Data3.xlsx",sheet_name=None)
csv=pd.read_csv(ROOT/"DataSet_main.csv",sep=";",decimal=",",engine="python")
datasets={"DATA.xls/2018":xls["2018"],"DATA.xls/2014":xls["2014"],"Data3.xlsx/2014-2017":d3["2014-2017"],"DataSet_main.csv":csv}


In [2]:
OUT=ROOT/"datasets"/"processed"; OUT.mkdir(parents=True,exist_ok=True)
reports=[]
for name,df in datasets.items():
    clean=df.copy(deep=True)
    # Parse decimal-comma numeric-looking object columns into a separate copy only when >=90% parse.
    for col in clean.select_dtypes(include="object").columns:
        parsed=pd.to_numeric(clean[col].astype(str).str.strip().str.replace(",",".",regex=False),errors="coerce")
        nonmissing=clean[col].notna().sum()
        if nonmissing and parsed.notna().sum()/nonmissing>=.90:
            clean[col+"__numeric_parse"] = parsed
    file=OUT/(name.replace("/","_").replace(".","_")+"_parsed_review.csv")
    clean.to_csv(file,index=False)
    reports.append({"dataset":name,"rows_source":len(df),"rows_exported":len(clean),"source_duplicate_rows":int(df.duplicated().sum()),"source_missing_cells":int(df.isna().sum().sum()),"export":str(file.relative_to(ROOT)),"imputation":"none; unsupported without definitions","exclusions":0})
report=pd.DataFrame(reports); display(report); report.to_csv(ROOT/"outputs"/"audit"/"preprocessing_log.csv",index=False)

,dataset,rows_source,rows_exported,source_duplicate_rows,source_missing_cells,export,imputation,exclusions
0,DATA.xls/2018,3658,3658,0,0,datasets/processed/DATA_xls_2018_parsed_review...,none; unsupported without definitions,0
1,DATA.xls/2014,3057,3057,74,19,datasets/processed/DATA_xls_2014_parsed_review...,none; unsupported without definitions,0
2,Data3.xlsx/2014-2017,2024,2024,15,142,datasets/processed/Data3_xlsx_2014-2017_parsed...,none; unsupported without definitions,0
3,DataSet_main.csv,876,876,1,46,datasets/processed/DataSet_main_csv_parsed_rev...,none; unsupported without definitions,0


## Decisions and leakage control
No observations were excluded. Parsing copies are additional columns with `__numeric_parse` suffixes; original columns are retained exactly. The CSV decimal comma is parsed by pandas at ingestion. Sentinel-looking values remain untouched because their meanings are undocumented. The processed folder contains review copies rather than analysis-ready clinical data. For future model comparisons, use `SimpleImputer` and scaling/encoding only inside a `Pipeline`/`ColumnTransformer` fitted on training folds.